In [4]:
import torch
import torch.nn as nn

In [49]:
num_features = 40
kernel_size_temporal = 30
kernel_size_spatial = 14
kernel_size_avg_pool = 15
ffn_embedding_dim = 80
vocab_size = 3

temporal_conv = nn.Conv2d(in_channels=1,
                        out_channels=num_features,
                        kernel_size=(1, kernel_size_temporal),
                        padding="same")

spatial_conv = nn.Conv2d(in_channels=num_features,
                                       out_channels=num_features,
                                       kernel_size=(kernel_size_spatial, 1),
                                       padding="valid")

batch_norm1 = nn.BatchNorm2d(num_features)
batch_norm2 = nn.BatchNorm2d(num_features)

elu = nn.ELU()

avg_pool = nn.AvgPool2d(kernel_size=(1, kernel_size_avg_pool))

fc1 = nn.Linear(2400, ffn_embedding_dim)
fc2 = nn.Linear(ffn_embedding_dim, vocab_size)

softmax = nn.Softmax(dim=1)

In [ ]:
x = torch.randn(32, 14, 900) # (N, C, T) where H is num_channels

x = x.unsqueeze(1) # (N, C, H, W) where C is 1 and H is num_channels

x = temporal_conv(x) # (32, 40, 14, 900)
x = batch_norm1(x) # (32, 40, 14, 900)
x = elu(x) # (32, 40, 14, 900)
print("before:", x.shape)

x = spatial_conv(x) # (32, 40, 1, 900)
x = batch_norm2(x) # (32, 40, 1, 900)
x = elu(x) # (32, 40, 1, 900)
print("after:", x.shape)

x = avg_pool(x) # (32, 40, 1, 60)
print("pooled:", x.shape)

x = torch.flatten(x, 1) # (32, 40 * 1 * 60) = (32, 2400)
print("flattened:", x.shape)

x = fc1(x)
x = fc2(x)
print("dense:", x.shape)

x = softmax(x) # (32, 3) - each row is a probability distribution across the 3 classes
print("softmax:", x.shape)
print(x)

before: torch.Size([32, 40, 14, 900])
after: torch.Size([32, 40, 1, 900])
pooled: torch.Size([32, 40, 1, 60])
flattened: torch.Size([32, 2400])
dense: torch.Size([32, 3])
softmax: torch.Size([32, 3])
tensor([[0.3550, 0.3672, 0.2777],
        [0.3364, 0.3392, 0.3244],
        [0.3280, 0.3372, 0.3349],
        [0.3284, 0.3742, 0.2974],
        [0.3475, 0.3473, 0.3053],
        [0.3252, 0.3600, 0.3148],
        [0.3108, 0.3688, 0.3204],
        [0.3055, 0.3832, 0.3113],
        [0.3128, 0.3759, 0.3113],
        [0.3243, 0.3395, 0.3362],
        [0.3281, 0.3714, 0.3005],
        [0.3243, 0.3547, 0.3210],
        [0.3440, 0.3627, 0.2933],
        [0.3435, 0.3412, 0.3153],
        [0.3441, 0.3244, 0.3315],
        [0.3378, 0.3528, 0.3094],
        [0.3327, 0.3474, 0.3200],
        [0.3385, 0.3796, 0.2819],
        [0.3427, 0.3637, 0.2937],
        [0.3679, 0.3507, 0.2813],
        [0.3462, 0.3631, 0.2907],
        [0.3227, 0.3488, 0.3285],
        [0.3051, 0.3602, 0.3347],
        [0.3359, 0